# ESMFold2 Reprediction

Re-fold **target + binder** complexes (and optionally the **binder alone**) with ESMFold2, using the *exact* UmbraCore reprediction backend that the production pipeline runs — `UmbraCore.reprediction.esmfold2_cli`. Nothing here re-implements folding or metric normalization; the notebook just calls the pipeline functions, so pLDDT / pTM / i_pTM / pAE / i_pAE come out on the same scale as a real campaign.

**Inputs**: a target PDB/CIF (one or more chains) + one or more binder sequences.
**Outputs**: a metrics table (CSV) and a predicted complex PDB per binder.

> Run this in your GPU/ESMFold2 environment (the one built with `envs/install_umbra_lean.sh`). It needs `torch` + CUDA and the Biohub `esm` + `transformers` ESMFold2 stack.

## 1. Locate the repo and check the environment

In [1]:
import os, sys, importlib

# Point this at the UmbraOne repo root (the folder containing `UmbraCore/`).
REPO_ROOT = os.environ.get("UMBRA_REPO_ROOT", os.path.abspath(".."))
if not os.path.isdir(os.path.join(REPO_ROOT, "UmbraCore")):
    # Fallback: walk up from CWD until we find UmbraCore/
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.isdir(os.path.join(p, "UmbraCore")):
            REPO_ROOT = p; break
        p = os.path.dirname(p)
assert os.path.isdir(os.path.join(REPO_ROOT, "UmbraCore")), f"UmbraCore/ not found under {REPO_ROOT!r}; set UMBRA_REPO_ROOT"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("REPO_ROOT =", REPO_ROOT)

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — folding will be very slow"))


REPO_ROOT = /scratch/nickel/UmbraOne_patched


torch 2.7.1+cu126 | CUDA available: True | NVIDIA L40S


## 2. Configuration

`ESMFOLD2` mirrors the `esmfold2` block of `UmbraDesign/settings_advanced/blocks/reprediction.json` (the production reprediction defaults). Change what you need:

- **`num_loops`** — refinement loops. Production reprediction uses `3` (Biohub cookbook standard). `20` is the old, slower baseline; `1` is quick-and-dirty.
- **`auto_select_model_by_msa`** — when `True`, uses `single_sequence_model_name` (`ESMFold2-Fast`) if no MSA is supplied and `msa_model_name` (`ESMFold2`) when one is. Set `False` to force `model_name`.
- **`target_msa`** — if `True`, tries to attach the target MSA (from `target_msa_path` or a TargetPrep JSON in `TARGET_SETTINGS`); otherwise single-sequence. Left `False` here for a self-contained run.
- **`seed`**, **`num_sampling_steps`**, **`num_diffusion_samples`** — diffusion sampling controls.

In [2]:
# ---- Target -------------------------------------------------------------
TARGET_PDB    = "/scratch/nickel/Reprediction_PD1_designs/Target/mPD1.pdb"     # PDB or CIF of the target (with real coordinates)
TARGET_CHAINS = "A"                          # comma-separated target chain id(s), e.g. "A" or "A,B"

# ---- Binder sequences to repredict --------------------------------------
# Either list them inline as (name, sequence) ...
BINDERS = [
    ]
# ... or load from a CSV with columns name,seq  (set to a path to use it):
BINDERS_CSV = "/scratch/nickel/Reprediction_PD1_designs/PD1_duo_DesignSummary.csv"   # e.g. "/path/to/binders.csv"

# ---- Output -------------------------------------------------------------
OUTPUT_DIR = os.path.abspath("./esmfold2_repredict_out")

# ---- ESMFold2 reprediction settings (mirrors reprediction.json) ---------
ESMFOLD2 = {
    "model_name": "biohub/ESMFold2",
    "single_sequence_model_name": "biohub/ESMFold2-Fast",
    "msa_model_name": "biohub/ESMFold2",
    "auto_select_model_by_msa": True,
    "device": "auto",
    "num_loops": 3,
    "num_sampling_steps": 50,
    "num_diffusion_samples": 1,
    "target_msa": True,          # set True + provide target_msa_path / TARGET_SETTINGS to condition on target MSA
    "binder_msa": False,
    "target_msa_path": None,
    "target_msa_paths": None,
    "binder_msa_path": None,
    "max_msa_sequences": 256,
    "seed": 0,
    "keep_batch_inputs": False,
}

# Optional: a TargetPrep/target settings dict (for target-MSA lookup). None is fine.
TARGET_SETTINGS = "/scratch/nickel/Reprediction_PD1_designs/Target/mPD1/mPD1.json"

# Fold the binder monomer too (Binder_pLDDT / Binder_pTM / Binder_pAE)?
PREDICT_BINDER_ALONE = False

# ---- On-the-fly target MSA (ColabFold MMseqs2 server) -------------------
# When True, the notebook searches an MSA for each target chain via the same
# ColabFold MMseqs2 API the pipeline uses (tools/colabfold_msa_server.py),
# writes one A3M per chain, and conditions reprediction on it. This SUBMITS
# each target chain sequence to the public server below.
COMPUTE_TARGET_MSA = False
MSA_SERVER = {
    "host_url": "https://api.colabfold.com",
    "use_env": True,          # include environmental (BFD/MGnify) databases
    "use_filter": True,
    "allow_online": True,     # bypass the UMBRA_NO_ONLINE_MSA kill switch for this call
    # Optional auth for a private/self-hosted MMseqs2 server:
    "msa_server_username": None,
    "msa_server_password": None,
    "api_key_header": None,
    "api_key_value": None,
}

# ---- On-the-fly BINDER MSA (same MMseqs2 server) ------------------------
# When True, search an MSA for each binder sequence too and condition the
# fold on it. Note: de novo binders usually return shallow/empty MSAs, so
# this often does little — enable it for natural/graft-derived binders.
# Because the backend applies one binder MSA per fold, candidates are folded
# one at a time (each with its own A3M) when this is on.
COMPUTE_BINDER_MSA = False


## 3. Assemble the pipeline inputs

Build the `candidates`, `advanced_settings`, and `design_paths` structures exactly as the pipeline expects them.

In [3]:
import csv, re

def load_binders():
    if BINDERS_CSV:
        rows = []
        with open(BINDERS_CSV) as fh:
            for r in csv.DictReader(fh):
                rows.append((r["name"].strip(), re.sub(r"[^A-Za-z]", "", r["sequence"]).upper()))
        return rows
    return [(n, re.sub(r"[^A-Za-z]", "", s).upper()) for n, s in BINDERS]

binders = load_binders()
assert binders, "No binder sequences provided"
candidates = [{"name": n, "seq": s} for n, s in binders]
for c in candidates:
    print(f"{c['name']:40s} len={len(c['seq'])}")

# advanced_settings just needs the backend flag + the esmfold2 block.
advanced_settings = {"prediction_backend": "esmfold2", "esmfold2": ESMFOLD2}

# design_paths: the backend writes complex PDBs to MPNN/ and monomers to MPNN/Binder/.
design_paths = {
    "MPNN":        os.path.join(OUTPUT_DIR, "MPNN"),
    "MPNN/Binder": os.path.join(OUTPUT_DIR, "MPNN", "Binder"),
}
for p in design_paths.values():
    os.makedirs(p, exist_ok=True)
failure_csv = os.path.join(OUTPUT_DIR, "failures.csv")

assert os.path.isfile(TARGET_PDB), f"TARGET_PDB not found: {TARGET_PDB}"


PD1_duo_l150_s59861_mpnn1                len=150
PD1_duo_l149_s38460_mpnn10               len=149
PD1_duo_l148_s75252_mpnn19               len=148
PD1_duo_l147_s78011_mpnn6                len=147
PD1_duo_l146_s60537_mpnn2                len=146
PD1_duo_l146_s4790_mpnn8                 len=146
PD1_duo_l146_s20240_mpnn9                len=146
PD1_duo_l144_s22658_mpnn1                len=144
PD1_duo_l144_s25328_mpnn2                len=144
PD1_duo_l142_s45966_mpnn2                len=142
PD1_duo_l142_s31803_mpnn16               len=142
PD1_duo_l141_s93450_mpnn2                len=141
PD1_duo_l140_s31721_mpnn8                len=140
PD1_duo_l138_s47892_mpnn16               len=138
PD1_duo_l138_s32967_mpnn19               len=138
PD1_duo_l136_s46826_mpnn8                len=136
PD1_duo_l136_s50942_mpnn9                len=136
PD1_duo_l136_s99931_mpnn4                len=136
PD1_duo_l135_s40505_mpnn3                len=135
PD1_duo_l134_s54007_mpnn2                len=134
PD1_duo_l134_s70048_

In [4]:
print("=== CONFIG SUMMARY ===")
print("model_name:", ESMFOLD2["model_name"])
print("single_sequence_model_name:", ESMFOLD2["single_sequence_model_name"])
print("msa_model_name:", ESMFOLD2["msa_model_name"])
print("auto_select_model_by_msa:", ESMFOLD2["auto_select_model_by_msa"])
print("target_msa:", ESMFOLD2["target_msa"], "| target_msa_path:", ESMFOLD2["target_msa_path"], "| target_msa_paths:", ESMFOLD2["target_msa_paths"])
print("binder_msa:", ESMFOLD2["binder_msa"], "| binder_msa_path:", ESMFOLD2["binder_msa_path"])
print("TARGET_SETTINGS:", TARGET_SETTINGS)
print("device:", ESMFOLD2["device"])
print("num_candidates:", len(candidates))


=== CONFIG SUMMARY ===
model_name: biohub/ESMFold2
single_sequence_model_name: biohub/ESMFold2-Fast
msa_model_name: biohub/ESMFold2
auto_select_model_by_msa: True
target_msa: True | target_msa_path: None | target_msa_paths: None
binder_msa: False | binder_msa_path: None
TARGET_SETTINGS: /scratch/nickel/Reprediction_PD1_designs/Target/mPD1/mPD1.json
device: auto
num_candidates: 96


## 4. (Optional) Compute the target MSA on the fly

Runs `run_mmseqs2_server` from `tools/colabfold_msa_server.py` — the pipeline's own MMseqs2 client — once for all target chains, writes an A3M per chain into `OUTPUT_DIR/target_msa/`, and points `ESMFOLD2["target_msa_paths"]` at them (keyed by chain id). With `auto_select_model_by_msa=True`, supplying an MSA also flips the model from `ESMFold2-Fast` to `ESMFold2`.

> ⚠️ This **submits each target chain sequence** to `MSA_SERVER["host_url"]` (public ColabFold server by default). Skipped entirely unless `COMPUTE_TARGET_MSA = True`. The A3M is cached in `OUTPUT_DIR/target_msa/`, so re-runs don't re-query.

In [5]:
if COMPUTE_TARGET_MSA:
    from tools.colabfold_msa_server import run_mmseqs2_server
    from UmbraCore.reprediction.esmfold2_cli import _target_sequences

    msa_dir = os.path.join(OUTPUT_DIR, "target_msa")
    os.makedirs(msa_dir, exist_ok=True)

    # (chain_id, sequence) for every target chain, in the same order/filtering
    # the reprediction backend uses.
    tgt_chain_seqs = _target_sequences(TARGET_PDB, TARGET_CHAINS)
    chain_ids = [cid for cid, _ in tgt_chain_seqs]
    chain_seqs = [seq for _, seq in tgt_chain_seqs]
    print("Searching MSA for target chains:", chain_ids)

    # One server round-trip for all chains; returns one A3M text per input seq.
    a3m_texts = run_mmseqs2_server(
        chain_seqs,
        prefix=os.path.join(msa_dir, "target"),
        **MSA_SERVER,
    )

    target_msa_paths = {}
    for cid, seq, a3m in zip(chain_ids, chain_seqs, a3m_texts):
        p = os.path.join(msa_dir, f"target_{cid}.a3m")
        with open(p, "w") as fh:
            fh.write(a3m)
        depth = sum(1 for ln in a3m.splitlines() if ln.startswith(">"))
        print(f"  chain {cid}: len={len(seq)} depth={depth} -> {p}")
        target_msa_paths[cid] = p

    # Wire the MSAs into the reprediction settings (mutates the shared dict).
    ESMFOLD2["target_msa"] = True
    ESMFOLD2["target_msa_paths"] = target_msa_paths
    print("Target MSA conditioning enabled:", target_msa_paths)
else:
    print("COMPUTE_TARGET_MSA is False; using target_msa =", ESMFOLD2["target_msa"])


COMPUTE_TARGET_MSA is False; using target_msa = True


## 5. (Optional) Compute binder MSAs on the fly

Searches one MSA **per binder sequence** (a single MMseqs2 round-trip for all binders), writes an A3M per candidate into `OUTPUT_DIR/binder_msa/`, and records `{name: a3m_path}`. Unlike the target MSA (one per chain, shared), each binder needs its own A3M matched to its own sequence — so §6/§7 fold candidates one at a time when this is enabled.

> ⚠️ Submits each **binder** sequence to `MSA_SERVER["host_url"]`. Skipped unless `COMPUTE_BINDER_MSA = True`. De novo binders typically get shallow/empty results.

In [6]:
binder_msa_paths = {}
if COMPUTE_BINDER_MSA:
    from tools.colabfold_msa_server import run_mmseqs2_server

    bmsa_dir = os.path.join(OUTPUT_DIR, "binder_msa")
    os.makedirs(bmsa_dir, exist_ok=True)

    names = [c["name"] for c in candidates]
    seqs = [c["seq"] for c in candidates]
    print("Searching MSA for", len(seqs), "binder(s)")

    a3m_texts = run_mmseqs2_server(
        seqs,
        prefix=os.path.join(bmsa_dir, "binder"),
        **MSA_SERVER,
    )
    for name, seq, a3m in zip(names, seqs, a3m_texts):
        safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", name)
        p = os.path.join(bmsa_dir, f"{safe}.a3m")
        with open(p, "w") as fh:
            fh.write(a3m)
        depth = sum(1 for ln in a3m.splitlines() if ln.startswith(">"))
        print(f"  {name:40s} len={len(seq)} depth={depth} -> {p}")
        binder_msa_paths[name] = p
    ESMFOLD2["binder_msa"] = True
else:
    print("COMPUTE_BINDER_MSA is False; binder folded single-sequence")


COMPUTE_BINDER_MSA is False; binder folded single-sequence


## 6. Repredict the target + binder complex

Calls `predict_binder_complex_esmfold2_batch` — the same function `UmbraCore.reprediction.pipeline` dispatches to when `prediction_backend == "esmfold2"`. It folds every candidate against the target, writes a complex PDB, and returns normalized metrics.

When `COMPUTE_BINDER_MSA` is on, candidates are folded one at a time so each gets its own binder A3M (the backend applies a single `binder_msa_path` per fold); otherwise all candidates go through one batch call.

In [7]:
from UmbraCore.reprediction.esmfold2_cli import (
    predict_binder_complex_esmfold2_batch,
    predict_binder_alone_esmfold2_batch,
)

def _fold_complex(cands):
    return predict_binder_complex_esmfold2_batch(
        candidates=cands,
        target_pdb=TARGET_PDB,
        target_chains=TARGET_CHAINS,
        length=len(cands[0]["seq"]),
        design_paths=design_paths,
        advanced_settings=advanced_settings,
        filters={},
        failure_csv=failure_csv,
        target_settings=TARGET_SETTINGS,
    )

complex_results = {}
if COMPUTE_BINDER_MSA and binder_msa_paths:
    for c in candidates:                       # one fold per candidate, own binder A3M
        ESMFOLD2["binder_msa_path"] = binder_msa_paths.get(c["name"])
        complex_results.update(_fold_complex([c]))
    ESMFOLD2["binder_msa_path"] = None         # reset shared config
else:
    complex_results = _fold_complex(candidates)

# complex_results: {name: (stats, ok)} where stats = {1: {pLDDT,pTM,i_pTM,pAE,i_pAE}}
resolved_model = ESMFOLD2["model_name"] if not ESMFOLD2["auto_select_model_by_msa"] else (
    ESMFOLD2["msa_model_name"] if (ESMFOLD2["target_msa"] or ESMFOLD2["binder_msa"]) else ESMFOLD2["single_sequence_model_name"]
)
print("=== RESOLVED MODEL (best-effort, replicating auto_select_model_by_msa logic) ===")
print(resolved_model)

print("=== FOLD RESULTS ===")
for name, (stats, ok) in complex_results.items():
    pdb_path = os.path.join(design_paths["MPNN"], f"{name}_model1.pdb")
    exists = os.path.isfile(pdb_path)
    size = os.path.getsize(pdb_path) if exists else 0
    print(f"{name:40s} ok={ok}  stats={stats.get(1)}  pdb_exists={exists} ({size} bytes)")


ESMC: transformer_engine is not installed; falling back to pure-PyTorch LayerNorm+Linear / LayerNorm+MLP. Outputs will differ numerically — measured on the unnormalized residual stream (before the final LayerNorm), ~O(10) max-diff in fp32 and ~O(100) in bf16; after the final LayerNorm these shrink to a few ULP and perplexity stays within rounding noise. Install with `pip install transformer-engine[pytorch]` to enable fused fp32-reduction LayerNorm.


🚨 No checkpoint found for ESMCForSequenceClassification.forward. Please add a `checkpoint` arg to `auto_docstring` or add one in ESMCConfig's docstring
🚨 No checkpoint found for ESMCForTokenClassification.forward. Please add a `checkpoint` arg to `auto_docstring` or add one in ESMCConfig's docstring


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Loading CCD dictionary from /home/nickel/.cache/huggingface/hub/models--biohub--ESMFold2/snapshots/8fc3ff471022fdce52c77030685eb775de0c00a3/ccd.pkl


=== RESOLVED MODEL (best-effort, replicating auto_select_model_by_msa logic) ===
biohub/ESMFold2
=== FOLD RESULTS ===
PD1_duo_l150_s59861_mpnn1                ok=True  stats={'pLDDT': 0.8571, 'pTM': 0.8368, 'i_pTM': 0.7681, 'pAE': 5.41, 'i_pAE': None}  pdb_exists=True (174798 bytes)
PD1_duo_l149_s38460_mpnn10               ok=True  stats={'pLDDT': 0.8398, 'pTM': 0.8141, 'i_pTM': 0.7016, 'pAE': 5.63, 'i_pAE': None}  pdb_exists=True (175203 bytes)
PD1_duo_l148_s75252_mpnn19               ok=True  stats={'pLDDT': 0.8561, 'pTM': 0.8295, 'i_pTM': 0.7214, 'pAE': 5.56, 'i_pAE': None}  pdb_exists=True (176580 bytes)
PD1_duo_l147_s78011_mpnn6                ok=True  stats={'pLDDT': 0.8513, 'pTM': 0.8855, 'i_pTM': 0.8792, 'pAE': 5.24, 'i_pAE': None}  pdb_exists=True (175446 bytes)
PD1_duo_l146_s60537_mpnn2                ok=True  stats={'pLDDT': 0.738, 'pTM': 0.6703, 'i_pTM': 0.6445, 'pAE': 9.85, 'i_pAE': None}  pdb_exists=True (177309 bytes)
PD1_duo_l146_s4790_mpnn8                 ok=True  sta

## 7. (Optional) Repredict the binder alone

Folds each binder as a monomer to get `Binder_pLDDT / Binder_pTM / Binder_pAE` — the monomer-confidence metrics used in filtering. Uses the same per-binder A3M from §5 when `COMPUTE_BINDER_MSA` is on.

In [8]:
mono_results = {}
if PREDICT_BINDER_ALONE:
    def _fold_mono(cands):
        return predict_binder_alone_esmfold2_batch(
            candidates=cands,
            length=len(cands[0]["seq"]),
            trajectory_pdb=TARGET_PDB,   # unused for folding; binder is folded from sequence
            binder_chain="B",
            prediction_models=[0],
            advanced_settings=advanced_settings,
            design_paths=design_paths,
        )
    if COMPUTE_BINDER_MSA and binder_msa_paths:
        for c in candidates:
            ESMFOLD2["binder_msa_path"] = binder_msa_paths.get(c["name"])
            mono_results.update(_fold_mono([c]))
        ESMFOLD2["binder_msa_path"] = None
    else:
        mono_results = _fold_mono(candidates)
    for name, stats in mono_results.items():
        print(f"{name:40s} {stats.get(1)}")


## 8. Collect results into a table

In [9]:
import pandas as pd

rows = []
for c in candidates:
    name = c["name"]
    stats, ok = complex_results.get(name, ({}, False))
    m = stats.get(1, {}) if isinstance(stats, dict) else {}
    row = {
        "name": name,
        "length": len(c["seq"]),
        "ok": ok,
        "pLDDT": m.get("pLDDT"),
        "pTM": m.get("pTM"),
        "i_pTM": m.get("i_pTM"),
        "pAE": m.get("pAE"),
        "i_pAE": m.get("i_pAE"),
        "complex_pdb": os.path.join(design_paths["MPNN"], f"{name}_model1.pdb"),
    }
    if PREDICT_BINDER_ALONE:
        bm = mono_results.get(name, {}).get(1, {})
        row.update({
            "Binder_pLDDT": bm.get("Binder_pLDDT"),
            "Binder_pTM": bm.get("Binder_pTM"),
            "Binder_pAE": bm.get("Binder_pAE"),
        })
    rows.append(row)

df = pd.DataFrame(rows)
out_csv = os.path.join(OUTPUT_DIR, "reprediction_metrics.csv")
df.to_csv(out_csv, index=False)
print("Wrote", out_csv)
df


Wrote /scratch/nickel/Reprediction_PD1_designs/ESMFold2/mouse/esmfold2_repredict_out/reprediction_metrics.csv


,name,length,ok,pLDDT,pTM,i_pTM,pAE,i_pAE,complex_pdb
0,PD1_duo_l150_s59861_mpnn1,150,True,0.8571,0.8368,0.7681,5.41,None,/scratch/nickel/Reprediction_PD1_designs/ESMFo...
1,PD1_duo_l149_s38460_mpnn10,149,True,0.8398,0.8141,0.7016,5.63,None,/scratch/nickel/Reprediction_PD1_designs/ESMFo...
2,PD1_duo_l148_s75252_mpnn19,148,True,0.8561,0.8295,0.7214,5.56,None,/scratch/nickel/Reprediction_PD1_designs/ESMFo...
3,PD1_duo_l147_s78011_mpnn6,147,True,0.8513,0.8855,0.8792,5.24,None,/scratch/nickel/Reprediction_PD1_designs/ESMFo...
4,PD1_duo_l146_s60537_mpnn2,146,True,0.7380,0.6703,0.6445,9.85,None,/scratch/nickel/Reprediction_PD1_designs/ESMFo...
...,...,...,...,...,...,...,...,...,...
91,PD1_duo_l81_s68430_mpnn2,81,True,0.8253,0.7921,0.7269,6.40,None,/scratch/nickel/Reprediction_PD1_designs/ESMFo...
92,PD1_duo_l81_s13409_mpnn7,81,True,0.6947,0.6162,0.2130,12.69,None,/scratch/nickel/Reprediction_PD1_designs/ESMFo...
93,PD1_duo_l81_s59351_mpnn5,81,True,0.7804,0.7266,0.5736,7.46,None,/scratch/nickel/Reprediction_PD1_designs/ESMFo...
94,PD1_duo_l80_s50417_mpnn2,80,True,0.8570,0.8687,0.8514,4.68,None,/scratch/nickel/Reprediction_PD1_designs/ESMFo...


## Notes

- **Metric scales** match the pipeline: pLDDT / pTM / i_pTM are 0–1; pAE / i_pAE are in the AF3-style 0–31.75 Å bin space (via `_normalize_pae`). This is what your filter thresholds are tuned against.
- **`i_pTM` / `i_pAE`** are the interface metrics — the ones to watch for binder quality against the target. In the `s820962` example above, expect low `i_pTM`, consistent with that trajectory being gated out during design.
- **Target MSA**: three ways — (a) `COMPUTE_TARGET_MSA = True` to search on the fly (§4); (b) point `ESMFOLD2["target_msa_path"]` / `target_msa_paths` at existing A3M(s); or (c) a `TARGET_SETTINGS` dict referencing a TargetPrep JSON.
- **Binder MSA**: set `COMPUTE_BINDER_MSA = True` (§5) to search one A3M per binder, or point `ESMFOLD2["binder_msa_path"]` at an existing A3M. Because the backend applies one binder MSA per fold, enabling it makes §6/§7 fold candidates one at a time. De novo binders usually gain little from an MSA.
- **Kill switch**: on-the-fly search honors `UMBRA_NO_ONLINE_MSA` — if that env var is set and you haven't passed `allow_online=True` (it's `True` by default in `MSA_SERVER`), the search refuses rather than contacting the server.
- **Model download**: `biohub/ESMFold2` and `biohub/ESMFold2-Fast` weights are pulled from HuggingFace on first use and cached; the model stays resident for the whole notebook session.
- **Lower-level path**: if you want to bypass the batch wrappers entirely, `_fold_sequences(sequences=[{id,sequence,msa}...], advanced_settings=...)` in the same module returns a raw ESMFold2 fold result (`.plddt`, `.ptm`, `.iptm`, `.pae`, `.ipae`, `.complex.to_mmcif()`).